# 03a · Generating Synthetic Weather & Price Paths

This notebook builds the **stochastic engine** behind the Monte Carlo project finance model. Its job is to fit probabilistic models to the historical data prepared in notebook `02` and then **simulate thousands of plausible 30-year futures** for the two variables that drive solar-project revenue:

1. **Solar resource** — represented by the daily **Clear-Sky Index (CSI)**.
2. **Electricity price** — the merchant market price the plant sells into.

The simulated paths are exported as a directory bundle under `data/PV_SYNTHETIC_DATA/` (a metadata JSON plus one CSV per simulation — see the *Export* section for why), which notebook `03b` (`pv-markov-model.ipynb`) feeds through the project finance model — one simulation at a time — to produce a full **distribution** of equity returns instead of the single deterministic answer from notebook `01`.

## Modelling approach

**Clear-Sky Index → Markov chain + Beta distributions.** Weather is *persistent*: a cloudy day is more likely to be followed by another cloudy day. A plain i.i.d. draw from the historical CSI histogram would miss this autocorrelation. So we use a **two-layer model**:

- A **3-state Markov chain** captures the day-to-day regime persistence (low / medium / high CSI). The transition probabilities are estimated from history.
- Within each state, a fitted **Beta distribution** generates the continuous CSI value (Beta is the natural choice for a bounded `[0, 1]` quantity).

**Electricity price → Kernel Density Estimate (KDE).** Daily merchant prices are sampled from a non-parametric **Gaussian KDE** fitted to recent history, which reproduces the empirical price distribution without assuming a parametric shape.

**Validation.** The final cells confirm that the synthetic CSI and price distributions match the historical ones they were fitted to.

In [1]:
import pandas as pd
from pvlib.clearsky import ineichen
from pvlib.location import Location
from pvlib import iotools
from datetime import datetime, timedelta, timezone
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy.stats import beta, johnsonsu, gaussian_kde

import plotly.io as pio
from utils import plotly_setup
from utils.custom_widgets import ProgressBarDark

import json

In [2]:
pio.templates.default = "theme_dark_1"

# Solar PV Data

In [3]:
filepath = r'data\PV_DATA_38.8128_-6.5211_20150101-20260101.csv'

df_solar = pd.read_csv(filepath, index_col=0, parse_dates=True)
df_solar = df_solar.iloc[1:]
df_solar

,clearsky_ghi,clearsky_dni,clearsky_dhi,ghi,dni,cloud_coverage
time,,,,,,
2015-01-01 01:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.60
2015-01-01 02:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.00
2015-01-01 03:00:00+00:00,0.0,0.0,0.0,0.0,0.0,1.79
2015-01-01 04:00:00+00:00,0.0,0.0,0.0,0.0,0.0,0.60
2015-01-01 05:00:00+00:00,0.0,0.0,0.0,0.0,0.0,1.19
...,...,...,...,...,...,...
2025-12-30 19:00:00+00:00,0.0,0.0,0.0,0.0,0.0,9.29
2025-12-30 20:00:00+00:00,0.0,0.0,0.0,0.0,0.0,10.99
2025-12-30 21:00:00+00:00,0.0,0.0,0.0,0.0,0.0,8.24


Calculate `clear_sky_index`as:

$$
CSI = \frac{GHI_{measured} + 0.5}{GHI_{clearsky} + 0.5}
$$

* $+0.5$ is added both in numerator and denominator to prevent division by zero errors and spike of `CSI`during sunrise and sunset

In [4]:
df_solar_daily = df_solar.resample('D').sum()

df_solar_daily['clear_sky_index'] = np.clip((df_solar_daily['ghi'] + 0.5)/(df_solar_daily['clearsky_ghi'] + 0.5), min=0, max=1)
df_solar_daily

,clearsky_ghi,clearsky_dni,clearsky_dhi,ghi,dni,cloud_coverage,clear_sky_index
time,,,,,,,
2015-01-01 00:00:00+00:00,2676.171761,6770.582240,296.666824,2861.80,6843.66,6.56,1.000000
2015-01-02 00:00:00+00:00,2688.313566,6782.464770,298.164777,2978.07,7082.35,634.81,1.000000
2015-01-03 00:00:00+00:00,2701.545986,6795.527655,299.751409,3015.75,7415.11,63.12,1.000000
2015-01-04 00:00:00+00:00,2715.868063,6810.913910,301.428454,3037.00,7458.46,87.62,1.000000
2015-01-05 00:00:00+00:00,2731.274230,6826.523618,303.194987,3033.11,7315.51,94.53,1.000000
...,...,...,...,...,...,...,...
2025-12-26 00:00:00+00:00,2627.746564,6727.176628,289.671761,2620.33,5332.27,609.67,0.997178
2025-12-27 00:00:00+00:00,2633.649895,6730.747810,290.659610,1697.51,2164.62,1372.33,0.644614
2025-12-28 00:00:00+00:00,2640.665806,6737.831215,291.739077,1549.80,2155.55,1704.70,0.586976


In [5]:
fig = go.Figure()


normalized_clearsky_ghi = df_solar_daily['clearsky_ghi']/df_solar_daily['clearsky_ghi'].max()
normalized_nasa_ghi = df_solar_daily['ghi']/df_solar_daily['ghi'].max()


fig.add_trace(
    go.Scatter(
        x=df_solar_daily.index,
        y=normalized_clearsky_ghi,
        name='clearsky_ghi'
    )
)

fig.add_trace(
    go.Scatter(
        x=df_solar_daily.index,
        y=normalized_nasa_ghi,
        name='ghi_nasa',
        mode='markers',
        marker=dict(
            size=3
        ),
    )
)


fig.add_trace(
    go.Scatter(
        x=df_solar_daily.index,
        y=df_solar_daily['clear_sky_index'],
        name='clear_sky_index',
        mode='markers',
        marker=dict(
            size=3
        ),
    )
)


fig.add_trace(
    go.Scatter(
        x=df_solar_daily.index,
        y=df_solar_daily['clear_sky_index'].rolling(window=100).mean(),
        name='average_csi',
        opacity=0.7
    )
)


fig.show()

In [6]:
fig = go.Figure()

bin_start, bin_end, bin_size = 0, 1, 0.05

fig.add_trace(
    go.Histogram(
        x=df_solar_daily['clear_sky_index'].values,
        xbins=dict(
            size=bin_size
        ),
        opacity=1,
    )
)

fig.show()

## Solar Markov States

We model the daily CSI as a **Markov-modulated Beta process**, built from two helper classes:

- **`BetaFunction`** — fits a Beta distribution to a sample of CSI values. Because Beta lives on `[0, 1]` but our per-state CSI values occupy only a sub-range, the data is first rescaled into `[0, 1]`, fitted, and the random samples are scaled back to the original bounds.
- **`MarkovState`** — represents one CSI regime, defined by `[lower, upper)` bounds and holding the `BetaFunction` fitted to the days that fall in that regime.

We then discretise the continuous CSI into **3 states** — `[0, 0.6)` (overcast), `[0.6, 0.9)` (intermediate), `[0.9, 1.0]` (clear) — and assign every historical day to one. From this labelled sequence we estimate:

1. the **transition probability matrix** `P[i, j]` = P(tomorrow in state *j* | today in state *i*), counted from consecutive days, and
2. a **Beta distribution per state**, capturing the spread of CSI values within each regime.

Together these let us generate new CSI sequences that preserve both the *persistence* (via the chain) and the *shape* (via the Beta fits) of the real data.

In [7]:
class BetaFunction():
    def __init__(self):
        pass

    def fit_data(self, data):
        buffer = 1e-5
        self.scale_min = min(data) - buffer
        self.scale_max = max(data) + buffer
        
        rescaled_data = (data - self.scale_min) / (self.scale_max - self.scale_min)
        alpha_param, beta_param, _, _ = beta.fit(rescaled_data, floc=0, fscale=1)
        self.alpha = alpha_param
        self.beta = beta_param

    def get_random_sample(self):
        if self.beta is None or self.beta is None:
            raise Exception("Beta fucntion must be calculated first to get a random sample")
        sample = beta.rvs(self.alpha, self.beta, size=1)[0]

        return float(self.scale_min + (sample * (self.scale_max - self.scale_min)))
    
    def get_random_samples(self, size):
        if self.beta is None or self.alpha is None:
            raise Exception("Beta function must be calculated first to get random samples")
        if size == 0:
            return np.array([])
        samples = beta.rvs(self.alpha, self.beta, size=size)
        return self.scale_min + (samples * (self.scale_max - self.scale_min))

In [8]:
class MarkovState():
    def __init__(self, lower, upper, lower_tolerance = 0, upper_tolerance = 0):
        self.lower = lower - lower_tolerance
        self.upper = upper + upper_tolerance
        self.beta_func = None

    def fit_beta(self, data):
        self.beta_func = BetaFunction()
        self.beta_func.fit_data(data)

    def __repr__(self):
        return f'Markov State ({self.lower:.2E} -> {self.lower:.2E})'

In [9]:
markov_states = [
    MarkovState(0.0, 0.6),
    MarkovState(0.6, 0.9),
    MarkovState(0.9, 1.0, upper_tolerance=1e-5),
]


conditions = [ (df_solar_daily['clear_sky_index'] >= ms.lower) & (df_solar_daily['clear_sky_index'] < ms.upper ) for ms in markov_states ]
states = list(range(len(markov_states)))
              
df_solar_daily['markov_state'] = np.select(conditions, states , default=0)
df_solar_daily

,clearsky_ghi,clearsky_dni,clearsky_dhi,ghi,dni,cloud_coverage,clear_sky_index,markov_state
time,,,,,,,,
2015-01-01 00:00:00+00:00,2676.171761,6770.582240,296.666824,2861.80,6843.66,6.56,1.000000,2
2015-01-02 00:00:00+00:00,2688.313566,6782.464770,298.164777,2978.07,7082.35,634.81,1.000000,2
2015-01-03 00:00:00+00:00,2701.545986,6795.527655,299.751409,3015.75,7415.11,63.12,1.000000,2
2015-01-04 00:00:00+00:00,2715.868063,6810.913910,301.428454,3037.00,7458.46,87.62,1.000000,2
2015-01-05 00:00:00+00:00,2731.274230,6826.523618,303.194987,3033.11,7315.51,94.53,1.000000,2
...,...,...,...,...,...,...,...,...
2025-12-26 00:00:00+00:00,2627.746564,6727.176628,289.671761,2620.33,5332.27,609.67,0.997178,2
2025-12-27 00:00:00+00:00,2633.649895,6730.747810,290.659610,1697.51,2164.62,1372.33,0.644614,1
2025-12-28 00:00:00+00:00,2640.665806,6737.831215,291.739077,1549.80,2155.55,1704.70,0.586976,0


### Estimating the transition matrix and per-state Beta fits

The next cell walks the historical state sequence day by day, tallying how often each state is followed by each other state, and normalises the counts row-wise into the **transition probability matrix** (each row sums to 1). The diagonal dominance you see — high values on `P[i, i]` — is the quantitative signature of weather persistence. The cell after it fits one **Beta distribution per state** and prints each state's bounds, frequency and fitted `alpha`/`beta` parameters.

In [10]:
states_values = df_solar_daily['markov_state'].to_numpy()
states_values_counts = np.zeros_like(states)

transition_matrix = np.zeros((len(markov_states), len(markov_states)))

for i, x in enumerate(states_values[1:], start=1):
    s0, s1 = states_values[i-1], states_values[i]
    transition_matrix[s0, s1] += 1
    states_values_counts[s0] += 1

states_frequencies = states_values_counts/states_values_counts.sum()
    
transition_probability_matrix = transition_matrix/transition_matrix.sum(1).reshape(transition_matrix.shape[0], 1)
transition_probability_matrix

array([[0.35283993, 0.46471601, 0.18244406],
       [0.22683142, 0.41394528, 0.3592233 ],
       [0.05169418, 0.17115552, 0.7771503 ]])

In [11]:
for state in states:
    markov_state = markov_states[state]
    csi_values = df_solar_daily[df_solar_daily['markov_state'] == state]['clear_sky_index'].values
    markov_state.fit_beta(csi_values)
    print(f'State: {state} \t Bounds: [ {markov_state.lower:.2f} , {markov_state.upper:.2f} ) \t Frequency: {states_frequencies[state]:.1%} \t Alpha: {markov_state.beta_func.alpha:.2E} \tBeta:  {markov_state.beta_func.beta:.2E}')

State: 0 	 Bounds: [ 0.00 , 0.60 ) 	 Frequency: 14.5% 	 Alpha: 2.61E+00 	Beta:  1.03E+00
State: 1 	 Bounds: [ 0.60 , 0.90 ) 	 Frequency: 28.2% 	 Alpha: 1.07E+00 	Beta:  8.85E-01
State: 2 	 Bounds: [ 0.90 , 1.00 ) 	 Frequency: 57.3% 	 Alpha: 8.03E-01 	Beta:  1.56E-01


# Electricity Market Price Data

We load Spanish day-ahead electricity prices (ESIOS) and reduce them to a single series suitable for a solar plant:

- **Sunny hours only** — filtered to `07:00–17:00`, since a solar plant only earns during daylight; the relevant price is the average price *while it is generating*, not the 24-hour average.
- **Recent history only** — restricted to **2024 onward**. This deliberately excludes the 2022 Ukraine-war energy-price spike and the abnormally low pre-COVID prices, both of which would bias the fitted distribution.
- **Daily mean** — resampled to one average daytime price per day, matching the daily resolution of the CSI model.

This cleaned daily price series is what the KDE in the synthetic-data step is fitted to.

In [12]:
pricefile = r'data\ESIOS_DATA_2015_2026.csv'
df_price = pd.read_csv(pricefile, index_col=0)
df_price.index = pd.to_datetime(df_price.index, utc=True)
df_price = df_price.rename_axis('time')
df_price = df_price.rename(columns={'value':'price'})
df_price = df_price[['price']]
df_price = df_price.between_time("07:00", "17:00")
df_price.head(3)

,price
time,
2015-01-01 07:00:00+00:00,32.4
2015-01-01 08:00:00+00:00,36.6
2015-01-01 09:00:00+00:00,43.1


Get the price data since 2023 to prevent skewing the model with the outlier spike of the ucraine war energy crisis and the pre-covid low volatility prices and calculate the daily mean for just the sunny hours

In [13]:
df_price:pd.DataFrame = df_price[df_price.index.year >= 2024]
df_price = df_price.between_time("07:00", "17:00").resample("D").mean()
df_price

,price
time,
2024-01-01 00:00:00+00:00,20.932727
2024-01-02 00:00:00+00:00,48.711818
2024-01-03 00:00:00+00:00,67.123636
2024-01-04 00:00:00+00:00,95.203636
2024-01-05 00:00:00+00:00,60.214545
...,...
2026-06-02 00:00:00+00:00,3.170976
2026-06-03 00:00:00+00:00,0.379512
2026-06-04 00:00:00+00:00,1.303171


In [14]:
fig = go.Figure()


fig.add_trace(
    go.Scatter(
        x=df_price.index,
        y=df_price['price'],
        name='Electricty Price (€/MWh)',
        line=dict(width=1),
    )
)

fig.add_trace(
    go.Scatter(
        x=df_price.index,
        y=df_price['price'].rolling(window=30).mean(),
        name='Electricty Price (€/MWh) (moving average 30)',
        line=dict(width=1),
    )
)


fig.update_layout(
    # Fix the Y-axis label overlapping numbers
    yaxis_title_text="Price (€/MWh)",
    yaxis_title_standoff=25,   
    margin=dict(l=60, r=20, t=40, b=50) 
)


fig.show()

# Synthetic Data

## Get solar radiation curves

First we generate the **deterministic backbone**: the clear-sky GHI for every day of a future **30-year** horizon (starting 2027). This is the predictable astronomical envelope — identical across all simulations. Each Monte Carlo path will then multiply this clear-sky curve by a *randomly simulated* daily CSI to obtain that scenario's actual irradiance.

In [15]:
lat, lon = 38.812821, -6.521058
start_date = datetime(2027, 1, 1, tzinfo=timezone.utc )
years = 30

In [16]:
loc = Location(lat, lon, tz=timezone.utc) 
end_date = datetime(start_date.year + years, 1, 1 , tzinfo=timezone.utc)
times = pd.date_range(start=start_date, end=end_date, freq='1h', tz='UTC')

# 2. Get Clear Sky GHI
df_synth = loc.get_clearsky(times)
df_synth = df_synth.rename(columns={"ghi":"clearsky_ghi", "dni":"clearsky_dni", "dhi":"clearsky_dhi"})
df_synth = df_synth.resample("D").sum()
df_synth

,clearsky_ghi,clearsky_dni,clearsky_dhi
2027-01-01 00:00:00+00:00,2669.958839,6728.623882,294.927493
2027-01-02 00:00:00+00:00,2682.163097,6742.950784,296.480955
2027-01-03 00:00:00+00:00,2695.488120,6758.841009,298.135149
2027-01-04 00:00:00+00:00,2709.930577,6776.253919,299.889284
2027-01-05 00:00:00+00:00,2725.486332,6795.143958,301.742373
...,...,...,...
2056-12-28 00:00:00+00:00,2639.047112,6696.770427,290.293026
2056-12-29 00:00:00+00:00,2647.539479,6705.702131,291.510268
2056-12-30 00:00:00+00:00,2657.156499,6716.287652,292.829113
2056-12-31 00:00:00+00:00,2667.897356,6728.506853,294.249502


## Generate synthetic data

The simulation loop runs **`n_simulations = 1000`** independent 30-year paths. For each path:

1. **Prices** — draw one daytime price per day directly from the fitted Gaussian KDE (`price_kde.resample`), clipped to a sane `[−50, 400] €/MWh` range.
2. **Weather regime** — walk the **Markov chain** day by day: pick the next state by comparing a uniform random draw against the cumulative transition probabilities of the current state. The first day is seeded from the long-run state frequencies.
3. **CSI values** — for each day, draw a continuous CSI from the **Beta distribution of that day's state** (sampled in bulk per state for speed).
4. **Assemble** — attach the simulated `csi` and `price` columns to a copy of the deterministic clear-sky backbone.

The result is a list of 1,000 daily DataFrames, each a self-consistent synthetic future combining a persistent weather sequence with an independent price sequence.

In [17]:
transition_cum_probs = np.cumsum(transition_probability_matrix, axis=1)
transition_cum_probs

array([[0.35283993, 0.81755594, 1.        ],
       [0.22683142, 0.6407767 , 1.        ],
       [0.05169418, 0.2228497 , 1.        ]])

In [18]:
n_simulations = 1000
synthetic_days = len(df_synth.index)

price_kde = gaussian_kde(df_price['price'].values, bw_method=0.05)
transition_cum_probs = np.cumsum(transition_probability_matrix, axis=1)
transition_cum_probs_list = transition_cum_probs.tolist()

simulations = []

sim_loop = ProgressBarDark(total=n_simulations, desc="Runnning simulations")

for n in sim_loop:

    # 2. Vectorized price sampling
    synthetic_price_values = price_kde.resample(synthetic_days).flatten()
    synthetic_price_values = np.clip(synthetic_price_values, -50, 400)

    # 3. Optimized Markov Chain state generation 
    state_sequence = np.zeros(synthetic_days, dtype=int)
    current_state = np.random.choice(states, p=states_frequencies)
    state_sequence[0] = current_state
    
    # Pre-generate uniform random values for state transition logic
    rands = np.random.rand(synthetic_days)
    
    for i in range(1, synthetic_days):
        p = transition_cum_probs_list[current_state]
        r = rands[i]

        # Dynamic routing independent of the number of states
        for idx, boundary in enumerate(p):
            if r < boundary:
                current_state = idx
                break
        state_sequence[i] = current_state

    # 4. Vectorized Beta sampling grouped by state frequencies
    synthetic_csi_values = np.zeros(synthetic_days)
    for s in states:
        mask = (state_sequence == s)
        count = np.sum(mask)
        if count > 0:
            # Generate all samples for state 's' at once
            synthetic_csi_values[mask] = markov_states[s].beta_func.get_random_samples(count)

    # 5. Populate the final DataFrame slice
    _df = df_synth.copy()
    _df['csi'] = synthetic_csi_values
    _df['price'] = synthetic_price_values

    simulations.append(_df)

Runnning simulations:   0%|          | 0/1000 [00:00<?, ?it/s]

## Export Synthetic Data

A single JSON holding all 1,000 simulations inline would be **~1 GB** — well past
GitHub's 100 MB per-file limit. Instead we write a small **directory bundle**:

```
data/PV_SYNTHETIC_DATA/
    PV_SYNTHETIC_DATA.json      # generation + fitted-model parameters, and the
                                # relative path of every simulation file
    simulations/
        simulation_0000.csv     # one CSV per Monte Carlo path (each small)
        simulation_0001.csv
        ...
```

Each simulation becomes its own modestly-sized CSV, and the JSON carries only metadata
(location, horizon, Markov/Beta parameters, transition matrix) plus the list of relative
paths. `pv-markov-model.ipynb` reads the JSON and then loads the CSVs it points to.

In [23]:
import os

# Output layout (avoids a single multi-GB JSON that would exceed GitHub's 100 MB
# per-file limit):
#
#   data/PV_SYNTHETIC_DATA/
#       PV_SYNTHETIC_DATA.json      <- parameters + relative paths to each simulation
#       simulations/
#           simulation_0000.csv
#           simulation_0001.csv
#           ...
#
# Each simulation is written as its own (small) CSV; the JSON only holds metadata
# and the list of relative paths.

export_dirname = 'PV_SYNTHETIC_DATA'
export_dir = os.path.join('data', export_dirname)
simulations_subdir = 'simulations'
simulations_dir = os.path.join(export_dir, simulations_subdir)

os.makedirs(simulations_dir, exist_ok=True)

simulation_files = []
export_loop = ProgressBarDark(total=len(simulations), desc="Exporting simulations data")

for i in export_loop:
    sim: pd.DataFrame = simulations[i]
    rel_path = f'{simulations_subdir}/simulation_{i:04d}.csv'
    sim.to_csv(os.path.join(export_dir, rel_path))
    simulation_files.append(rel_path)

print(f'Wrote {len(simulation_files)} simulation CSVs to {simulations_dir}')

Exporting simulations data:   0%|          | 0/1000 [00:00<?, ?it/s]

Wrote 1000 simulation CSVs to data\PV_SYNTHETIC_DATA\simulations


In [24]:
# Metadata JSON: generation parameters + the fitted stochastic-model parameters
# + the relative path of every simulation CSV.
metadata = {
    'parameters': {
        'lat': lat,
        'lon': lon,
        'start_date': start_date.strftime('%Y-%m-%d'),
        'years': years,
        'n_simulations': n_simulations,
        'synthetic_days': synthetic_days,
        'price_clip': [-50, 400],
        'markov_states': [
            {
                'state': s,
                'lower': markov_states[s].lower,
                'upper': markov_states[s].upper,
                'frequency': float(states_frequencies[s]),
                'beta_alpha': markov_states[s].beta_func.alpha,
                'beta_beta': markov_states[s].beta_func.beta,
            }
            for s in states
        ],
        'transition_probability_matrix': transition_probability_matrix.tolist(),
        'state_frequencies': states_frequencies.tolist(),
    },
    'simulations': simulation_files,
}

export_filepath = os.path.join(export_dir, f'{export_dirname}.json')

with open(export_filepath, 'w') as f:
    json.dump(metadata, f, indent=4)

print(f'Wrote metadata for {len(simulation_files)} simulations to {export_filepath}')

Wrote metadata for 1000 simulations to data\PV_SYNTHETIC_DATA\PV_SYNTHETIC_DATA.json


## Synthetic CSI Data Validation

As a sanity check, we overlay the **synthetic** CSI distribution on the **historical** one. A good match confirms that the Markov + Beta model faithfully reproduces the empirical CSI distribution it was fitted to — so the simulated weather is statistically representative of the real site.

In [21]:
fig = go.Figure()

bin_start, bin_end, bin_size = 0, 2, 0.05

fig.add_trace(
    go.Histogram(
        x=df_solar_daily['clear_sky_index'].values,
        xbins=dict(
            size=bin_size
        ),
        opacity=0.75,
        histnorm='probability',
        name='historic_csi_values'
    )
)


fig.add_trace(
    go.Histogram(
        x=synthetic_csi_values,
        xbins=dict(
            size=bin_size
        ),
        opacity=0.75,
        histnorm='probability',
        name='synthetic_csi_values'
    )
)

fig.update_layout(
    title="Clear Sky Index",
    barmode='overlay', # Options: 'overlay', 'stack', 'group'
    bargap=0.02,
    xaxis=dict(
        title="CSI"
    ),
    yaxis=dict(
        tickformat=".1%"
    )
)

fig.show()

## Synthetic Price Data Validation

The same validation for **price**: the KDE-sampled synthetic prices should track the historical daytime-price histogram. With both CSI and price validated, the exported `PV_SYNTHETIC_DATA/` bundle is ready to drive the Monte Carlo project finance model in `pv-markov-model.ipynb`.

In [22]:
fig = go.Figure()

bin_size = 5

fig.add_trace(
    go.Histogram(
        x=df_price['price'].values,
        xbins=dict(
            size=bin_size
        ),
        opacity=0.75,
        histnorm='probability',
        name='historic_price_values'
    )
)


fig.add_trace(
    go.Histogram(
        x=synthetic_price_values,
        xbins=dict(
            size=bin_size
        ),
        opacity=0.75,
        histnorm='probability',
        name='synthetic_price_values'
    )
)


fig.update_layout(
    title="Price",
    barmode='overlay', # Options: 'overlay', 'stack', 'group'
    bargap=0.02,
    xaxis=dict(
        title="Price (€/MWh)"
    ),
    yaxis=dict(
        tickformat=".1%"
    )
)

fig.show()